In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [1] Imports & Motor-CAD 연결
# ─────────────────────────────────────────────────────────────────────────────
import pathlib
import sys
import importlib
from pathlib import Path
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat, savemat

# Repo root on path
repo_root = pathlib.Path.cwd().resolve()
while not ((repo_root / "tools").exists() or (repo_root / "tool").exists()) and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# 패키지 reload (의존순서: model → plot → parse → facade)
import tools.motorCAD.pyMCAD.magnetic_model as _mm
import tools.motorCAD.pyMCAD.magnetic_plot as _mp
import tools.motorCAD.pyMCAD.magnetic_parse as _mparse
import tools.motorCAD.pyMCAD.magnetic as _mag
importlib.reload(_mm)
importlib.reload(_mp)
importlib.reload(_mparse)
importlib.reload(_mag)

from tools.motorCAD.pyMCAD import (
    get_magnetic_timeseries_from_file,
    mcad_default_export_dir,
    find_latest_mes,
    list_mes_files,
)
from tools.motorCAD.pyMCAD.magnetic_model import MagElement

import ansys.motorcad.core as pymotorcad

# Motor-CAD 연결
mcad = pymotorcad.MotorCAD(open_new_instance=False)
refMotFilePath=r"D:\KangDH\Thesis\e10\refModel\e10Turn6V261.mot"
HalfSCMotFilePath=r"D:\KangDH\Thesis\e10\SLFEA_Half\e10Turn6V261SLFEA_Half.mot"
SCMotFilePath=r"D:\KangDH\Thesis\e10\SLFEA\e10Turn6V261SLFEA.mot"

# mcad.load_from_file(refMotFilePath)
print("Motor-CAD connected")
# print(f"  MOT file: {mcad.get_variable('CurrentMotFilePath_MotorLAB')}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [2] e10 모터 파라미터 설정
# ─────────────────────────────────────────────────────────────────────────────

# --- Conductor geometry (hairpin, rectangular) ---
# COND_WIDTH_MM = 2.5        # tangential width b [mm] (확인 필요 → Motor-CAD에서)
# COND_HEIGHT_MM = 2.5       # radial height h [mm] (확인 필요 → Motor-CAD에서)
SIGMA_CU = 5.8e7           # Cu conductivity @ 20°C [S/m]
# ACTIVE_LENGTH_MM = 100.0   # axial stack length [mm] (확인 필요)

# Motor-CAD에서 실제 값 읽기
try:
    COND_WIDTH_MM = float(mcad.get_variable("Copper_Width"))  # 슬롯 폭 / 병렬 수
    COND_HEIGHT_MM = float(mcad.get_variable("Copper_Height"))
    ACTIVE_LENGTH_MM = float(mcad.get_variable("Stator_Lam_Length"))
    n_parallel = int(mcad.get_variable("ParallelPaths"))
    n_turns = int(mcad.get_variable("MagTurnsConductor"))
    print(f"  Conductor: {COND_WIDTH_MM:.2f} x {COND_HEIGHT_MM:.2f} mm")
    print(f"  Active length: {ACTIVE_LENGTH_MM:.1f} mm")
    print(f"  Parallel paths: {n_parallel}, Turns/conductor: {n_turns}")
except Exception as e:
    print(f"  [WARN] Motor-CAD variable read failed: {e}")
    print(f"  Using default values: {COND_WIDTH_MM} x {COND_HEIGHT_MM} mm, L={ACTIVE_LENGTH_MM} mm")

# Convert to SI
b_m = COND_WIDTH_MM * 1e-3     # conductor tangential width [m]
h_m = COND_HEIGHT_MM * 1e-3    # conductor radial height [m]
L_a = ACTIVE_LENGTH_MM * 1e-3  # active length [m]

# --- Operating conditions ---
POLE_PAIRS = 4                 # 8-pole motor
SPEED_LIST = [2000, 4000, 16000]  # RPM

# Electrical frequency per speed
def speed_to_fe(speed_rpm, pole_pairs=POLE_PAIRS):
    return pole_pairs * speed_rpm / 60.0

print(f"\n  Speed → f_e: {[(s, f'{speed_to_fe(s):.0f} Hz') for s in SPEED_LIST]}")

# --- Skin depth & ξ table ---
MU_0 = 4 * np.pi * 1e-7
print(f"\n{'Speed [RPM]':>12} {'f_e [Hz]':>10} {'δ [mm]':>10} {'ξ = h/δ':>10}")
print("-" * 50)
for spd in SPEED_LIST:
    fe = speed_to_fe(spd)
    delta = 1.0 / np.sqrt(np.pi * fe * MU_0 * SIGMA_CU)
    xi_val = h_m / delta
    print(f"{spd:>12} {fe:>10.0f} {delta*1e3:>10.2f} {xi_val:>10.3f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [3] FEA 설정 + B-field TXT Export (속도별)
# ─────────────────────────────────────────────────────────────────────────────
# 
# 옵션 A: Motor-CAD를 여기서 직접 실행
# 옵션 B: 이미 실행된 결과의 .mes를 로드하여 export만 수행
#
# 여기서는 옵션 A (실행 + export) 를 기본으로 합니다.
# 이미 결과가 있으면 DO_SOLVE=False로 설정하세요.

DO_SOLVE = True
PHASE_ADVANCE = 43.33
RMS_CURRENT = 460  # Ref model

# FEA export 설정
FIRST_STEP = 1
FINAL_STEP = mcad.get_variable("TorquePointsPerCycle")  # Motor-CAD 기본 TorquePointsPerCycle 정도
EXPORT_COLUMNS = "RegCode,Bx,By,A,J,Je,Hx,Hy,Mur"

out_root = Path(mcad_default_export_dir(mcad))
export_dir = out_root / "ACLossCalcExport"
export_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [3] 90-Point FEA Sweep & Directory Backup
# ─────────────────────────────────────────────────────────────────────────────
import os
import shutil
from pathlib import Path
from datetime import datetime

# 90-Point Sweep Definition
CURRENT_LIST = np.linspace(0.1, 460.0, 5)   # 5 currents
PHASE_LIST = np.linspace(0.0, 90.0, 6)      # 6 phase angles
# SPEED_LIST is defined as [2000, 4000, 16000] in cell 1

mcad.set_variable("ProximityLossModel", 1)  # Hybrid mode

# FEA export settings (defined in cell 2)
FIRST_STEP = 1
FINAL_STEP = int(mcad.get_variable("TorquePointsPerCycle"))
EXPORT_COLUMNS = "RegCode,Bx,By,A,J,Je,Hx,Hy,Mur"

out_root = Path(mcad_default_export_dir(mcad))
backup_root = out_root / "ACLossCalcExport_Map"
backup_root.mkdir(parents=True, exist_ok=True)

sweep_results = []
total_points = len(SPEED_LIST) * len(CURRENT_LIST) * len(PHASE_LIST)
point_idx = 0

print(f"Starting 90-point sweep (speeds: {SPEED_LIST}, currents: {list(np.round(CURRENT_LIST, 1))}, phases: {list(np.round(PHASE_LIST, 1))})...")
print(f"Backup root: {backup_root}\n")

for speed in SPEED_LIST:
    mcad.set_variable("ShaftSpeed", speed)
    for current in CURRENT_LIST:
        mcad.set_variable("RMSCurrent", current)
        for phase in PHASE_LIST:
            mcad.set_variable("PhaseAdvance", phase)
            
            point_idx += 1
            print(f"[{point_idx}/{total_points}] Speed: {speed} RPM, Current: {current:.1f} A, Phase: {phase:.1f} deg")
            
            # 1. Run calculation
            print("  → Solving Hybrid FEA...")
            mcad.do_magnetic_calculation()
            
            # 2. Get latest solved results directory
            try:
                latest_mes = find_latest_mes(mcad)
                active_results_dir = latest_mes.parent
            except Exception as e:
                print(f"  [ERROR] Failed to locate latest .mes file: {e}")
                continue
            
            # 3. Create destination folder named after point variables
            point_folder_name = f"Speed_{speed}RPM_{current:.1f}A_{phase:.1f}deg"
            dest_point_dir = backup_root / point_folder_name
            dest_results_dir = dest_point_dir / "FEResultsData"
            
            # 4. Copy active results folder (contains OnLoadTorque_result_1.mes, etc.)
            print(f"  → Backing up results folder to: {point_folder_name}/FEResultsData")
            if dest_results_dir.exists():
                shutil.rmtree(dest_results_dir)
            shutil.copytree(active_results_dir, dest_results_dir)
            
            # 5. Export B-field TXT file to the destination directory
            txt_path = dest_point_dir / "FEA_data.txt"
            print(f"  → Exporting B-field TXT to: {point_folder_name}/FEA_data.txt")
            mcad.save_fea_data(str(txt_path), FIRST_STEP, FINAL_STEP, EXPORT_COLUMNS, "", ",")
            
            # 6. Read losses for summary
            try:
                total_w = float(mcad.get_variable("ACLoss_Hybrid_Total"))
                prox_w = float(mcad.get_variable("ACLoss_Hybrid_Prox_Total"))
                skin_w = float(mcad.get_variable("ACLoss_Hybrid_SkinEffect_Total"))
            except Exception as e:
                total_w, prox_w, skin_w = 0.0, 0.0, 0.0
                print(f"  [WARN] Failed to read hybrid losses: {e}")
                
            sweep_results.append({
                "speed": speed,
                "current": current,
                "phase": phase,
                "total_loss": total_w,
                "prox_loss": prox_w,
                "skin_loss": skin_w,
                "backup_dir": str(dest_point_dir)
            })
            print(f"  → Loss Summary: Total={total_w:.1f} W, Prox={prox_w:.1f} W, Skin={skin_w:.1f} W\n")

print(f"\n✓ Sweep complete! {point_idx} points processed and backed up under {backup_root}")